# Visualization and Analysis of Satellite-Derived Chlorophyll Data

This notebook processes and visualizes chlorophyll-a concentration time series derived from multiple satellite platforms (Landsat, Sentinel-2, and MODIS).

## Overview
- **Purpose**: Process, clean, and visualize chlorophyll/NDCI time series from satellite data
- **Input**: CSV files generated from Google Earth Engine processing
- **Output**: Time series plots showing chlorophyll-a trends over time

## Key Processing Steps:
1. **Data Import**: Read CSV files with date, NDCI/chlorophyll values
2. **NDCI Conversion**: Convert NDCI to chlorophyll-a concentration (Landsat/Sentinel)
3. **Outlier Removal**: Apply Median Absolute Deviation (MAD) method
4. **Visualization**: Generate time series plots for trend analysis

## Datasets:
- **Landsat**: 30m resolution, 16-day revisit, NDCI-based
- **Sentinel-2**: 10m resolution, 5-day revisit, NDCI-based
- **MODIS**: 500m resolution, daily, direct chlorophyll algorithm

In [ ]:
"""
Import required libraries for data processing and visualization.

Libraries:
- pandas: Data manipulation and CSV reading
- matplotlib: Plotting and visualization
- pathlib: File path handling
- numpy: Numerical operations and outlier detection
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS FOR DATA PROCESSING AND VISUALIZATION
# ============================================================================

def read_data(inpath):
    """
    Read satellite data from CSV file and prepare for analysis.
    
    Args:
        inpath: Path to CSV file containing satellite data
    
    Returns:
        DataFrame with parsed dates and sorted chronologically
    
    Processing:
    - Converts 'date' column to datetime format
    - Sorts data chronologically
    - Handles invalid date formats gracefully
    """
    df = pd.read_csv(inpath)
    # Parse dates with error handling for invalid formats
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df.sort_values('date', inplace=True)
    return df

def NDCI_to_chlorophyll(df, slope, offset):
    """
    Convert NDCI values to chlorophyll-a concentration.
    
    Args:
        df: DataFrame with 'ndci' column
        slope: Linear conversion factor (µg/L per NDCI unit)
        offset: Y-intercept of conversion equation
    
    Returns:
        DataFrame with added 'chl' column
    
    Algorithm:
        Chl-a = (NDCI × slope) + offset
    
    Note: This is an empirical conversion that may need calibration
    for specific water bodies.
    """
    df['chl'] = df['ndci'] * slope + offset
    return df

def remove_outliers(df, value_col='chl', threshold=5.0):
    """
    Remove outliers using Median Absolute Deviation (MAD) method.
    
    Args:
        df: Input DataFrame
        value_col: Column name for outlier detection (default: 'chl')
        threshold: Z-score threshold for outlier removal (default: 5.0)
    
    Returns:
        DataFrame with outliers removed
    
    Method:
    The MAD method is robust to outliers and works well for non-normal
    distributions common in environmental data:
    1. Calculate median of the data
    2. Calculate MAD = median(|x - median|)
    3. Convert MAD to robust standard deviation (σ = 1.4826 × MAD)
    4. Remove points with |z-score| > threshold
    
    The factor 1.4826 makes MAD consistent with standard deviation
    for normally distributed data.
    """
    median = df[value_col].median()
    mad = np.median(np.abs(df[value_col] - median))
    
    # Handle case where all values are identical (MAD = 0)
    if mad == 0:
        return df.copy()
    
    # Convert MAD to robust standard deviation estimate
    robust_sd = 1.4826 * mad
    
    # Calculate z-scores and filter outliers
    z_scores = np.abs(df[value_col] - median) / robust_sd
    cleaned_df = df[z_scores < threshold].copy()
    
    return cleaned_df

def plot_data(df, label, title, ylabel, save_filename=None):
    """
    Create time series plot of chlorophyll data and optionally save to file.
    
    Args:
        df: DataFrame with 'date' and 'chl' columns
        label: Legend label for the data series
        title: Plot title
        ylabel: Y-axis label
        save_filename: If provided, save figure to this filename (PNG format)
    
    Returns:
        matplotlib axes object for further customization
    
    Plot features:
    - 12x6 inch figure size for good visibility
    - Grid for easier value reading
    - Custom blue color for water-related data
    - Continuous line style for time series
    - High DPI (300) for publication-quality saved figures
    """
    # Create figure and axes
    fig, axes = plt.subplots(1, 1, figsize=(12, 6), sharex=True)
    
    # Define plot styling
    marker = ''           # No markers for cleaner look with many points
    linestyle = '-'       # Solid line for time series
    rgb = (60, 50, 255)   # Custom blue color
    linecolor = '#{:02X}{:02X}{:02X}'.format(*rgb)
    
    # Plot the time series
    axes.plot(df['date'], df['chl'], 
              marker=marker, 
              linestyle=linestyle, 
              color=linecolor, 
              label=label)
    
    # Configure plot appearance
    axes.set_title(title)
    axes.set_ylabel(ylabel)
    axes.grid(True, alpha=0.3)  # Light grid
    
    # Adjust layout to prevent label cutoff
    fig.tight_layout()
    
    # Save figure if filename provided
    if save_filename:
        # Ensure filename ends with .png
        if not save_filename.endswith('.png'):
            save_filename += '.png'
        
        # Save with high DPI for publication quality
        fig.savefig(save_filename, dpi=300, bbox_inches='tight')
        print(f"Figure saved as: {save_filename}")
    
    return axes

## Landsat Analysis - Detroit Lake

### Dataset Characteristics:
- **Temporal Coverage**: 2011-2025
- **Spatial Resolution**: 30m
- **Revisit Time**: 16 days (8 days with Landsat 7+8 combined)
- **Processing**: NDCI converted to chlorophyll-a using linear relationship

In [ ]:
"""
Process and visualize Landsat chlorophyll data for Detroit Lake.

This cell demonstrates the complete workflow from raw NDCI values
to cleaned chlorophyll time series visualization.
"""

# Define input CSV filename
csv_filename = 'Detroit_Landsat_NDCI_500m.csv'
png_filename = csv_filename.replace('.csv', '.png')

# Load Landsat NDCI data from CSV
df = read_data(csv_filename)

# Convert NDCI to chlorophyll-a concentration
# Empirical conversion parameters (site-specific calibration recommended)
offset = 0    # Baseline chlorophyll level (µg/L)
slope = 80    # Conversion factor (µg/L per NDCI unit)
df = NDCI_to_chlorophyll(df, slope, offset)

# Optional: Remove outliers using MAD method
# Uncomment to apply outlier removal with 5-sigma threshold
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Generate time series plot and save to PNG
label = 'Landsat Chl-a'
title = 'Detroit Lake - Landsat Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel, save_filename=png_filename)

## Landsat Analysis - Upper Klamath Lake

### Dataset Characteristics:
- **Lake Type**: Large shallow eutrophic lake
- **Known Issues**: High algae blooms, particularly cyanobacteria
- **Monitoring Importance**: Critical for water quality management
- **Data Quality**: May show higher variability due to shallow depth and wind mixing

In [ ]:
"""
Process and visualize Landsat chlorophyll data for Upper Klamath Lake.

Upper Klamath Lake typically shows higher chlorophyll concentrations
and more seasonal variability compared to Detroit Lake.
"""

# Define input CSV filename
csv_filename = 'Klamath_Landsat_NDCI_500m.csv'
png_filename = csv_filename.replace('.csv', '.png')

# Load Landsat NDCI data from CSV
df = read_data(csv_filename)

# Convert NDCI to chlorophyll-a concentration
# Same conversion parameters as Detroit Lake for consistency
offset = 0    # Baseline chlorophyll level (µg/L)
slope = 80    # Conversion factor (µg/L per NDCI unit)
df = NDCI_to_chlorophyll(df, slope, offset)

# Optional: Remove outliers using MAD method
# May be particularly useful for Upper Klamath due to extreme bloom events
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Generate time series plot and save to PNG
label = 'Landsat Chl-a'
title = 'Upper Klamath Lake - Landsat Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel, save_filename=png_filename)

## Sentinel-2 Analysis - Detroit Lake

### Dataset Characteristics:
- **Temporal Coverage**: 2015-2025 (Sentinel-2A launch: June 2015)
- **Spatial Resolution**: 10m (3x better than Landsat)
- **Revisit Time**: 5 days with both satellites
- **Advantages**: Higher resolution and frequency for detailed bloom monitoring
- **Red-Edge Band**: Specifically designed for vegetation/algae detection

In [ ]:
"""
Process and visualize Sentinel-2 chlorophyll data for Detroit Lake.

Sentinel-2 provides more frequent observations than Landsat,
allowing better capture of rapid bloom dynamics.
"""

# Define input CSV filename
csv_filename = 'Detroit_S2_NDCI_500m.csv'
png_filename = csv_filename.replace('.csv', '.png')

# Load Sentinel-2 NDCI data from CSV
df = read_data(csv_filename)

# Convert NDCI to chlorophyll-a concentration
# Using same parameters for cross-sensor comparison
offset = 0    # Baseline chlorophyll level (µg/L)
slope = 80    # Conversion factor (µg/L per NDCI unit)
df = NDCI_to_chlorophyll(df, slope, offset)

# Optional: Remove outliers using MAD method
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Generate time series plot and save to PNG
label = 'Sentinel-2 Chl-a'
title = 'Detroit Lake - Sentinel-2 Chlorophyll Time Series (2015-2025)'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel, save_filename=png_filename)

## Sentinel-2 Analysis - Upper Klamath Lake

### Dataset Characteristics:
- **High Data Volume**: More scenes available due to lake size and importance
- **Bloom Detection**: Red-edge band particularly effective for cyanobacteria
- **Seasonal Patterns**: Clear summer bloom peaks typically visible
- **Management Tool**: Used for real-time bloom warnings and advisories

In [ ]:
"""
Process and visualize Sentinel-2 chlorophyll data for Upper Klamath Lake.

The high frequency of Sentinel-2 observations is particularly valuable
for Upper Klamath Lake's dynamic algae bloom conditions.
"""

# Define input CSV filename
csv_filename = 'Klamath_S2_NDCI_500m.csv'
png_filename = csv_filename.replace('.csv', '.png')

# Load Sentinel-2 NDCI data from CSV
df = read_data(csv_filename)

# Convert NDCI to chlorophyll-a concentration
offset = 0    # Baseline chlorophyll level (µg/L)
slope = 80    # Conversion factor (µg/L per NDCI unit)
df = NDCI_to_chlorophyll(df, slope, offset)

# Optional: Remove outliers using MAD method
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Generate time series plot and save to PNG
label = 'Sentinel-2 Chl-a'
title = 'Upper Klamath Lake - Sentinel-2 Chlorophyll Time Series (2015-2025)'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel, save_filename=png_filename)

## MODIS Analysis - Detroit Lake

### Dataset Characteristics:
- **Temporal Coverage**: Daily observations (weather permitting)
- **Spatial Resolution**: 500m (single pixel covers significant lake area)
- **Dual Satellites**: Terra (morning) and Aqua (afternoon) provide 2x daily coverage
- **Algorithm**: Direct chlorophyll retrieval using green:red ratio
- **Trade-offs**: Lower spatial resolution but excellent temporal coverage

### Data Processing Notes:
- Outlier removal is essential due to residual cloud contamination
- Terra and Aqua may show systematic differences due to overpass times
- Daily data enables detection of rapid bloom onset/decline

In [ ]:
"""
Process and visualize MODIS chlorophyll data for Detroit Lake.

MODIS provides direct chlorophyll estimates (not NDCI-based) with
daily temporal resolution from both Terra and Aqua satellites.
"""

# Define input CSV and output PNG filenames
aqua_csv = 'Detroit_MODIS_Aqua_500m_Chl_singlePixel.csv'
terra_csv = 'Detroit_MODIS_Terra_500m_Chl_singlePixel.csv'
aqua_png = aqua_csv.replace('.csv', '.png')
terra_png = terra_csv.replace('.csv', '.png')

# Load MODIS data from both satellites
aqua_df = read_data(aqua_csv)
terra_df = read_data(terra_csv)

# Apply outlier removal - critical for MODIS due to:
# - Residual cloud effects
# - Atmospheric correction errors
# - Sun glint contamination
aqua_df_clean = remove_outliers(aqua_df, 'chl', threshold=5.0)
terra_df_clean = remove_outliers(terra_df, 'chl', threshold=5.0)

# Plot MODIS-Aqua (afternoon overpass ~1:30 PM local time) and save
label = 'MODIS-Aqua Chl-a'
title = 'Detroit Lake - MODIS-Aqua Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_aqua = plot_data(aqua_df_clean, label, title, ylabel, save_filename=aqua_png)

# Plot MODIS-Terra (morning overpass ~10:30 AM local time) and save
label = 'MODIS-Terra Chl-a'
title = 'Detroit Lake - MODIS-Terra Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_terra = plot_data(terra_df_clean, label, title, ylabel, save_filename=terra_png)

# Note: Differences between Terra and Aqua may indicate:
# - Diurnal variability in algae vertical distribution
# - Different sun angle effects
# - Sensor calibration differences

## MODIS Analysis - Upper Klamath Lake

### Dataset Characteristics:
- **Temporal Coverage**: Daily observations (weather permitting)
- **Spatial Resolution**: 500m (single pixel covers significant lake area)
- **Dual Satellites**: Terra (morning) and Aqua (afternoon) provide 2x daily coverage
- **Algorithm**: Direct chlorophyll retrieval using green:red ratio
- **Trade-offs**: Lower spatial resolution but excellent temporal coverage

### Data Processing Notes:
- Outlier removal is essential due to residual cloud contamination
- Terra and Aqua may show systematic differences due to overpass times
- Daily data enables detection of rapid bloom onset/decline

In [ ]:
"""
Process and visualize MODIS chlorophyll data for Upper Klamath Lake.

MODIS provides direct chlorophyll estimates (not NDCI-based) with
daily temporal resolution from both Terra and Aqua satellites.
"""

# Define input CSV and output PNG filenames
aqua_csv = 'Klamath_MODIS_Aqua_500m_Chl_singlePixel.csv'
terra_csv = 'Klamath_MODIS_Terra_500m_Chl_singlePixel.csv'
aqua_png = aqua_csv.replace('.csv', '.png')
terra_png = terra_csv.replace('.csv', '.png')

# Load MODIS data from both satellites
aqua_df = read_data(aqua_csv)
terra_df = read_data(terra_csv)

# Apply outlier removal - critical for MODIS due to:
# - Residual cloud effects
# - Atmospheric correction errors
# - Sun glint contamination
aqua_df_clean = remove_outliers(aqua_df, 'chl', threshold=5.0)
terra_df_clean = remove_outliers(terra_df, 'chl', threshold=5.0)

# Plot MODIS-Aqua (afternoon overpass ~1:30 PM local time)
label = 'MODIS-Aqua Chl-a'
title = 'Klamath Lake - MODIS-Aqua Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_aqua = plot_data(aqua_df_clean, label, title, ylabel, save_filename=aqua_png)

# Plot MODIS-Terra (morning overpass ~10:30 AM local time)
label = 'MODIS-Terra Chl-a'
title = 'Klamath Lake - MODIS-Terra Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_terra = plot_data(terra_df_clean, label, title, ylabel, save_filename=terra_png)

# Note: Differences between Terra and Aqua may indicate:
# - Diurnal variability in algae vertical distribution
# - Different sun angle effects
# - Sensor calibration differences